# EDA: cleaning

Import necessary functions

In [0]:
import pyspark.pandas as ps
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

Load necessary tables

In [0]:
activiteiten = spark.table("bronze.vcs_bi_zis_tijdschrijven_zpm")
ozp = spark.table("bronze.vcs_bi_zis_ozp")
verblijf = spark.table("bronze.vcs_bi_zis_verblijf")
diensten = spark.table("bronze.vcs_bi_personeel_dienst")
productiviteit = spark.table("bronze.vcs_bi_personeel_productiviteit")
fte_bezetting = spark.table("bronze.vcs_bi_personeel_bezetting")
fte_verzuim = spark.table("bronze.vcs_bi_personeel_verzuim")

The tariff list is imported from the [NZA website](https://puc.overheid.nl/nza/doc/PUC_800921_22/). This dataset will be used to standardize the monetary values.

In [0]:
# Define the full path to your Excel file inside the Volume
excel_path = "/Volumes/ggzingeest_test/default/zvw_tarievenlijst_c_2026_correct/Tarievenlijst_C_2026.xlsx"

# Inspect the Excel file to see all available sheet names
excel_file = pd.ExcelFile(excel_path)
print(f"Available sheets in the Excel file: {excel_file.sheet_names}\n")

# Read the data from the Excel file
pdf_tariffs = pd.read_excel(excel_path, sheet_name="opbouw tarieven 2026")

# Fix mixed-type columns: convert columns that should be numeric but contain text like 'vrij'
for col in pdf_tariffs.select_dtypes(include='object').columns:
    converted = pd.to_numeric(pdf_tariffs[col], errors='coerce')
    # If at least some values converted successfully, treat as numeric column
    if converted.notna().sum() > pdf_tariffs[col].notna().sum() * 0.5:
        pdf_tariffs[col] = converted

# 4. Convert the Pandas DataFrame to a Spark DataFrame
tariffs_2026_df = spark.createDataFrame(pdf_tariffs.astype(str))

# 5. Clean up column names to ensure they are lowercase and have no spaces
for col_name in tariffs_2026_df.columns:
    clean_name = col_name.strip().lower().replace(" ", "_")
    tariffs_2026_df = tariffs_2026_df.withColumnRenamed(col_name, clean_name)

print(f"✅ Excel loaded successfully! Total rows: {tariffs_2026_df.count()}")

# Show a preview of the data
display(tariffs_2026_df)

**Step 1:** Filter columns for all datasets, columns are hand-picked

In [0]:
df_activiteiten = activiteiten.select(
    "contactnummer", 
    "contactcode", 
    "contactomschrijving", 
    "rapportagedatum", 
    "begintijd", 
    "eindtijd", 
    "medewerker", 
    "beroepscategorie_code",
    "beroepscategorie",
    "beroepcode",
    "medewerkerberoep", 
    "org_level_1", 
    "org_level_2", 
    "org_level_3", 
    "org_level_4", 
    "tijd_client_direct_minuten", 
    "tijd_client_indirect_minuten", 
    "tijd_client_reis_minuten", 
    "tijd_client_minuten", 
    "tijd_medewerker_direct_minuten", 
    "tijd_medewerker_indirect_minuten",
    "indirecte_tijd_norm",
    "norm_directe_uren",
    "declaratiecode", 
    "declaratiecode_omschrijving", 
    "declaratiecode_categorie", 
    "groepsgrootte", 
    "bedrag_totaal", 
    "bedrag_totaal_nza", 
    "setting_code", 
    "setting",
    "soort_ggz_zorg", 
    "consult_type", 
    "type_financier", 
    "financier", 
    "client_code", 
    "no_show", 
    "declarabel", 
    "zorgtraject", 
    "diagnose", 
    "diagnosecode", 
    "diagnosegroep", 
    "diagnosegroepcode", 
    "meer_dan_365_dagen_zvw_verblijf", 
    "cruciale_zorg", 
    "zorgprogramma_omschrijving", 
    "zorgvraagtypering"
    )

df_ozp = ozp.select(
    "contactnummer", 
    "contactcode", 
    "contactomschrijving", 
    "rapportagedatum", 
    "begintijd", 
    "eindtijd", 
    "medewerker", 
    "beroepscategorie_code",
    "beroepscategorie",
    "beroepcode",
    "medewerkerberoep", 
    "org_level_1", 
    "org_level_2", 
    "org_level_3", 
    "org_level_4", 
    "tijd_client_direct_minuten", 
    "tijd_client_indirect_minuten", 
    "tijd_client_reis_minuten", 
    "tijd_client_minuten", 
    "tijd_medewerker_direct_minuten", 
    "tijd_medewerker_indirect_minuten", 
    "declaratiecode", 
    "declaratiecode_omschrijving", 
    "declaratiecode_categorie", 
    "bedrag_totaal", 
    "bedrag_totaal_nza", 
    "soort_ggz_zorg", 
    "type_financier", 
    "financier", 
    "client_code", 
    "no_show", 
    "crisis_binnen_budget", 
    "declarabel", 
    "meer_dan_365_dagen_zvw_verblijf", 
    "cruciale_zorg", 
    "zorgvraagtypering", 
    "zorgtraject"
    )    

df_verblijf = verblijf.select(
    "verblijfsdagnummer", 
    "zorgtraject", 
    "rapportagedatum", 
    "activiteitcode",
    "declaratiecode", 
    "declaratiecode_omschrijving", 
    "declaratiecode_categorie", 
    "financier", 
    "type_financier", 
    "declarabel", 
    "org_level_1", 
    "org_level_2", 
    "org_level_3", 
    "org_level_4", 
    "client_code", 
    "soort_ggz_zorg",  
    "beveiligde_setting", 
    "cruciale_zorg", 
    "meer_dan_365_dagen_zvw_verblijf", 
    "zorgvraagtypering", 
    "type_zvw_zorg", 
    "bedrag_totaal", 
    "bedrag_totaal_nza"
    )

df_productiviteit = productiviteit.select(
    "rapportagedatum", 
    "medewerker", 
    "personeelsnummer", 
    "geslacht", 
    "dienstverband_begindatum", 
    "vast_dienstverband", 
    "functie", 
    "functie_groep", 
    "beroepscategorie_code",
    "beroepscategorie",
    "leeftijd", 
    "leeftijdscategorie_per_5_jaar", 
    "leeftijdscategorie_per_10_jaar", 
    "in_loondienst", 
    "org_level_1", 
    "org_level_2", 
    "org_level_3", 
    "org_level_4", 
    "behandelfractie", 
    "contracturen_bruto", 
    "ziek_uren_excl_zwangerschap", 
    "zwangerschapsverlof_uren", 
    "verlof_uren_ouderschap", 
    "verlof_uren_ouderschap_betaald", 
    "verlof_uren_geboorte", 
    "verlof_uren_geboorte_aanvullend", 
    "verlof_uren_onbetaald", 
    "verlof_uren_zorg", 
    "verlof_uren_pleegzorg", 
    "verlof_uren_adoptie", 
    "verlof_uren_excl_ouderschap_of_onbetaald", 
    "bijzonder_verlof", 
    "opleiding_uren_gecorrigeerd", 
    "or_uren_gecorrigeerd", 
    "overige_neventaken_uren_gecorrigeerd", 
    "verlofsparen_uren", 
    "verlofrecht_uren", 
    "productieve_uren", 
    "declarabele_uren", 
    "declarabele_uren_totaal", 
    "declarabele_uren_direct", 
    "declarabele_uren_indirect",  
    "declarabele_uren_indirect_norm_zpm", 
    "declarabele_uren_reistijd", 
    "declarabele_uren_overig", 
    "contracturen_netto", 
    "contracturen_netto_behandelfractie", 
    "roosteruren_netto_behandelfractie", 
    "uren_bruto", "uren_netto", 
    "niet_inzetbare_uren", 
    "aantal_uren_per_fte"
    )

df_diensten = diensten.select(
    "dienst_uren", 
    "dienst_soort", 
    "dienst_groep", 
    "flexpool_inzet", 
    "werkzame_uren", 
    "betaalbare_uren", 
    "directe_uren", 
    "dienst_hoofdsoort", 
    "org_level_1", 
    "org_level_2", 
    "org_level_3", 
    "org_level_4", 
    "medewerker", 
    "personeelsnummer", 
    "rapportagedatum", 
)    

join_keys = [
    "rapportagedatum",
    "medewerker",
    "functie",
    "functie_groep",
    "beroepscategorie_code",
    "beroepscategorie",
    "org_level_1",
    "org_level_2",
    "org_level_3",
    "org_level_4",
    "org_level_5"
]

fte_verzuim_filtered = fte_verzuim.select(*(join_keys + ["fte_ziek", "fte_ziek_vangnet"]))
fte_joined = fte_bezetting.join(fte_verzuim_filtered, on=join_keys, how="left")

df_fte = fte_joined.select(
    "rapportagedatum",
    "medewerker",
    "personeelsnummer",
    "geslacht",
    "aantal_jaren_in_dienst",
    "aantal_dagen_in_dienst",
    "functie",
    "functie_groep",
    "beroepscategorie_code",
    "beroepscategorie",
    "beroep",
    "stagiair",
    "leeftijd",
    "in_loondienst",
    "org_level_1",
    "org_level_2",
    "org_level_3",
    "org_level_4",
    "org_level_5",
    "kostenplaats_code",
    "dagen_in_week",
    "dagen_in_maand",
    "contracturen_netto",
    "uren_bruto",
    "uren_netto",
    "fte_bruto_contract",
    "fte_netto_contract",
    "fte_ziek",
    "fte_ziek_vangnet"
)

**Step 2:** Inspect the shape and types and transform if necessary

In [0]:
# Show shape of the dataframes
print("Shape of df_activiteiten = " + str((df_activiteiten.count(), len(df_activiteiten.columns))))
print("Shape of df_ozp = " + str((df_ozp.count(), len(df_ozp.columns))))
print("Shape of df_verblijf = " + str((df_verblijf.count(), len(df_verblijf.columns))))
print("Shape of df_productiviteit = " + str((df_productiviteit.count(), len(df_productiviteit.columns))))
print("Shape of df_diensten = " + str((df_diensten.count(), len(df_diensten.columns))))
print("Shape of tariffs_2026_df = " + str((tariffs_2026_df.count(), len(tariffs_2026_df.columns))))
print("Shape of df_fte = " + str((df_fte.count(), len(df_fte.columns))))

In [0]:
# Show datatypes of activiteiten
df_activiteiten.printSchema()

In [0]:
# transform datatypes and/or add columns
df_activiteiten = df_activiteiten.withColumn(
    "begintijd", 
    F.regexp_replace(F.col("begintijd"), "^24:00.*", "23:59:59")
)

df_activiteiten = df_activiteiten.withColumn(
    "eindtijd", 
    F.regexp_replace(F.col("eindtijd"), "^24:00.*", "23:59:59")
)

# Calculate seconds over 24:00 (end-time is engineered as 25:15 on the same date, timestamp doesn't work on that)
tijd_delen = F.split(F.col("eindtijd"), ":")
totaal_seconden = (
    tijd_delen.getItem(0).cast("int") * 3600 + # Uren naar seconden
    tijd_delen.getItem(1).cast("int") * 60 +   # Minuten naar seconden
    F.coalesce(tijd_delen.getItem(2).cast("int"), F.lit(0)) # Seconden (met een vangnet als ze missen)
)

# Timestamp 
df_activiteiten = df_activiteiten.withColumn(
    "start_timestamp",
    F.to_timestamp(F.concat(F.col("rapportagedatum"), F.lit(" "), F.col("begintijd")), "yyyy-MM-dd HH:mm:ss")
)
df_activiteiten = df_activiteiten.withColumn(
    "end_timestamp",
    F.from_unixtime(
        F.unix_timestamp(F.to_date("rapportagedatum")) + totaal_seconden
    ).cast("timestamp")
)

# boolean
bool_columns = [
    'no_show',
    'declarabel',
    'meer_dan_365_dagen_zvw_verblijf',
    'cruciale_zorg',
]

for c in bool_columns:
    df_activiteiten = df_activiteiten.withColumn(c, F.when(F.col(c) == "Ja", True).otherwise(False))

# add columns year, week, month
df_activiteiten = (df_activiteiten
    .withColumn("jaar", F.year("rapportagedatum"))
    .withColumn("maand", F.month("rapportagedatum"))
    .withColumn("week", F.weekofyear("rapportagedatum"))
)

df_activiteiten.printSchema()

In [0]:
# Show datatypes of ozp
df_ozp.printSchema()

In [0]:
# transform datatypes and/or add columns
df_ozp = df_ozp.withColumn(
    "begintijd", 
    F.regexp_replace(F.col("begintijd"), "^24:00.*", "23:59:59")
)

df_ozp = df_ozp.withColumn(
    "eindtijd", 
    F.regexp_replace(F.col("eindtijd"), "^24:00.*", "23:59:59")
)

# Calculate seconds over 24:00 (end-time is engineered as 25:15 on the same date, timestamp doesn't work on that)
tijd_delen = F.split(F.col("eindtijd"), ":")
totaal_seconden = (
    tijd_delen.getItem(0).cast("int") * 3600 + # Uren naar seconden
    tijd_delen.getItem(1).cast("int") * 60 +   # Minuten naar seconden
    F.coalesce(tijd_delen.getItem(2).cast("int"), F.lit(0)) # Seconden (met een vangnet als ze missen)
)

# Timestamp (Met coalesce vangnet voor verblijfsdagen zonder tijd)
df_ozp = df_ozp.withColumn(
    "start_timestamp",
    F.coalesce(
        F.to_timestamp(F.concat(F.col("rapportagedatum"), F.lit(" "), F.col("begintijd")), "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp(F.col("rapportagedatum")) # Vangnet: zet op 00:00:00 van de rapportagedatum
    )
)

df_ozp = df_ozp.withColumn(
    "end_timestamp",
    F.coalesce(
        F.from_unixtime(
            F.unix_timestamp(F.to_date("rapportagedatum")) + totaal_seconden
        ).cast("timestamp"),
        F.to_timestamp(F.col("rapportagedatum")) # Vangnet: zet op 00:00:00 van de rapportagedatum
    )
)

# boolean
bool_columns = [
    'no_show',
    'declarabel',
    'meer_dan_365_dagen_zvw_verblijf',
    'cruciale_zorg',
    'crisis_binnen_budget'
]

for c in bool_columns:
    df_ozp = df_ozp.withColumn(c, F.when(F.col(c) == "Ja", True).otherwise(False))

# add columns year, week, month
df_ozp = (df_ozp
    .withColumn("jaar", F.year("rapportagedatum"))
    .withColumn("maand", F.month("rapportagedatum"))
    .withColumn("week", F.weekofyear("rapportagedatum"))
)

df_ozp.printSchema()

In [0]:
# Show datatypes of verblijf
df_verblijf.printSchema()
df_verblijf.select("beveiligde_setting","cruciale_zorg").distinct().show()

In [0]:
# boolean
bool_columns = [
    'declarabel',
    'meer_dan_365_dagen_zvw_verblijf',
    'cruciale_zorg',
    'beveiligde_setting'
]

for c in bool_columns:
    df_verblijf = df_verblijf.withColumn(c, F.when(F.col(c) == "Ja", True).otherwise(False))

# add columns year, week, month
df_verblijf = (df_verblijf
    .withColumn("jaar", F.year("rapportagedatum"))
    .withColumn("maand", F.month("rapportagedatum"))
    .withColumn("week", F.weekofyear("rapportagedatum"))
)

df_verblijf.printSchema()

In [0]:
# Filter activiteitcodes on codes for bed occupancy
df_verblijf.select("activiteitcode").distinct().show()

In [0]:
# Show datatypes of productiviteit
df_productiviteit.printSchema()

In [0]:
# Check columns for boolean values
df_productiviteit.select(
'vast_dienstverband',
'in_loondienst'
).distinct().show()

# change column to boolean value
df_productiviteit = df_productiviteit.withColumn("in_loondienst", F.when(F.col("in_loondienst") == "Ja", True).otherwise(False))

df_productiviteit.printSchema()

In [0]:
# Show datatypes of productiviteit
df_diensten.printSchema()

In [0]:
# Check columns for boolean values
df_diensten.select(
'flexpool_inzet',
'werkzame_uren',
'betaalbare_uren'
).distinct().show()

# Change column to boolean value
df_diensten = df_diensten.withColumn("flexpool_inzet", F.when(F.col("flexpool_inzet") == "Ja", True).otherwise(False))
df_diensten = df_diensten.withColumn("werkzame_uren", F.when(F.col("werkzame_uren") == "Ja", True).otherwise(False))
df_diensten = df_diensten.withColumn("betaalbare_uren", F.when(F.col("betaalbare_uren") == "Ja", True).otherwise(False))

df_diensten.printSchema()

In [0]:
# Show datatypes for tariffs_2026_df
tariffs_2026_df.printSchema()

In [0]:
dec_columns = [
    'kostprijs_prijspeil_2023',
    'kostprijs_prijspeil_2026',
    'check_indexatie_kostprijs',
    'kostprijs_inclusief_opslag_vgrev',
    'kostprijs_inclusief_opslag_vgrev_en_werkkapitaal',
    'kostprijs_inclusief_opslagen',
    'check_kostprijs_inclusief_index_en_opslagen',
    'kapitaallasten_prijspeil_2026',
    'kapitaallasten_inclusief_opslag_vgrev',
    'kapitaallasten_inclusief_opslag_vgrev_en_opslag_werkkapitaal',
    'kapitaallasten_inclusief_opslagen',
    'check_kapitaallasten_inclusief_opslagen',
    'tarief_prijspeil_2026',
    'check_tarief',
    'tijdrange'
]

# Convert string columns to double, handling 'nan' strings and comma decimals
for col_name in dec_columns:
    tariffs_2026_df = tariffs_2026_df.withColumn(
        col_name,
        F.when(
            (F.col(col_name) == 'nan') | (F.col(col_name) == 'None') | (F.col(col_name) == ''),
            F.lit(None)
        ).otherwise(
            F.regexp_replace(F.col(col_name), ',', '.').cast('double')
        )
    )

tariffs_2026_df.printSchema()

In [0]:
df_fte.printSchema()

**Step 4:** Check missing values for all tables


In [0]:
# define function for missing values in spark
def check_missing_values(df):
    total_rows = df.count()
    
    # We bouwen de lijst met berekeningen slim op basis van het datatype
    exprs = []
    for c, dtype in df.dtypes:
        if dtype in ('double', 'float'):
            # Bij kommagetallen checken we op NULL en NaN
            condition = F.col(c).isNull() | F.isnan(F.col(c))
        else:
            # Bij alle andere types (Date, String, etc.) checken we alleen op NULL
            condition = F.col(c).isNull()
            
        # Voeg de logica toe aan onze lijst met expressies
        exprs.append(F.sum(F.when(condition, 1).otherwise(0)).alias(c))
    
    # Voer de hele scan in één keer uit
    null_counts = df.select(exprs).collect()[0].asDict()
    
    # Maak het overzichtelijke Pandas tabelletje
    summary_data = []
    for col_name, missing_count in null_counts.items():
        percentage = (missing_count / total_rows) * 100 if total_rows > 0 else 0
        summary_data.append({
            "kolomnaam": col_name,
            "missing_values": missing_count,
            "percentage_missing": round(percentage, 2)
        })
    
    summary_df = pd.DataFrame(summary_data).sort_values(by="missing_values", ascending=False).reset_index(drop=True)
    
    return summary_df

In [0]:
check_missing_values(df_activiteiten)
# many missing values for certain columns, futher research to determine what has to be done

In [0]:
display(
    df_activiteiten.filter((F.col("setting_code").isNull()) & (F.col("declarabel") == True))
    .select("declaratiecode_categorie").distinct()
)

# Output shows that either declaratiecode_categorie is null or Groep, which has no setting. Therefore it can be filled with Unknown

# Fill all textcolumns with "Unknown" and all value columns with 0
df_activiteiten = df_activiteiten.fillna("Unknown").fillna(0)

# Check again
check_missing_values(df_activiteiten)

In [0]:
check_missing_values(df_ozp)
# further analysis is necessary

In [0]:
# tijd_medewerker_direct_minuten and tijd_medewerker_indirect_minuten are empty for 80%. However, they are very important for calculating the total time spent on the patient, so the columns will stay and be filed with 0
# Fill all textcolumns with "Unknown" and all value columns with 0
df_ozp = df_ozp.fillna("Unknown").fillna(0)

In [0]:
check_missing_values(df_ozp)


In [0]:
check_missing_values(df_verblijf)

In [0]:
# Drop verblijf_op_separeer and fill other columns
df_verblijf = (df_verblijf
    .drop("verblijf_op_separeer")
    .fillna("Unknown")
    .fillna(0)
)

# Check missing values again
check_missing_values(df_verblijf)

In [0]:
check_missing_values(df_productiviteit)

In [0]:
# Drop only the 100% empty columns and fill other columns
df_productiviteit = (df_productiviteit
    .drop("verlof_uren_pleegzorg")
    .drop("verlof_uren_geboorte_aanvullend")
    .drop("verlof_uren_geboorte")
    .drop("verlof_uren_ouderschap_betaald")
    .drop("verlof_uren_adoptie")
    .fillna("Unknown")
    .fillna(0)
)

# Check again
check_missing_values(df_productiviteit)

In [0]:
check_missing_values(df_diensten)

In [0]:
check_missing_values(df_fte)

In [0]:
df_fte = (df_fte
    .fillna("Unknown")
    .fillna(0)
)

check_missing_values(df_fte)

In [0]:
# Add column fte_inzet
df_fte = df_fte.withColumn("fte_inzet", F.col("fte_netto_contract") - F.col("fte_ziek_vangnet"))

**Step 5:** Check duplicates for all tables

In [0]:
def check_duplicates(df, subset_columns=None):
    """
    Checks for duplicates in a PySpark DataFrame.
    If 'subset_columns' is not provided, it checks for exact duplicate rows across all columns.
    If a list of columns is provided (e.g., ["client_id", "date"]), it checks for duplicates based on that specific combination.
    """
    # 1. Determine which columns to group by
    group_columns = subset_columns if subset_columns else df.columns
    
    # 2. Calculate the number of duplicates
    total_rows = df.count()
    unique_rows = df.dropDuplicates(subset=group_columns).count()
    duplicate_count = total_rows - unique_rows
    
    print(f"Total number of rows: {total_rows}")
    print(f"Number of duplicate rows: {duplicate_count}")

    # 3. Show the duplicate rows if any exist
    if duplicate_count > 0:
        print("Here is an overview of ALL rows that are duplicates (grouped together):")
        
        # Maak een 'Window' aan: we kijken naar groepen op basis van de gekozen kolommen
        window_spec = Window.partitionBy(*group_columns)
        
        # Tel het aantal keer dat de rij voorkomt binnen dat Window, filter > 1, en sorteer
        df_all_duplicates = (df.withColumn("freq_count", F.count("*").over(window_spec))
                               .filter(F.col("freq_count") > 1)
                               .orderBy(F.desc("freq_count"), *group_columns))
        
        display(df_all_duplicates)
    else:
        print("No duplicates found")

In [0]:
check_duplicates(df_activiteiten)

In [0]:
check_duplicates(df_ozp)
# Remove exact duplicates
df_ozp = df_ozp.dropDuplicates()
check_duplicates(df_ozp)

In [0]:
check_duplicates(df_verblijf)

In [0]:
check_duplicates(df_productiviteit)
# Remove exact duplicates
df_productiviteit = df_productiviteit.dropDuplicates()
check_duplicates(df_productiviteit)

In [0]:
check_duplicates(df_diensten)
# Remove exact duplicates
df_diensten = df_diensten.dropDuplicates()
check_duplicates(df_diensten)

In [0]:
check_duplicates(df_fte)
# remove exact duplicates
df_fte = df_fte.dropDuplicates()
check_duplicates(df_fte)

**Step 6:** Descriptive analytics

In [0]:
# It's handy to combine the dataframes into a dictionary, because the next steps will be done for all tables
dfs = {
    "Activiteiten": df_activiteiten,
    "OZP": df_ozp,
    "Verblijf": df_verblijf,
    "Productiviteit": df_productiviteit,
    "Diensten": df_diensten,
    "tarieven": tariffs_2026_df,
    "fte": df_fte
}

In [0]:
# Check numerical values
for name, df in dfs.items():
    print(f"\n{'='*40}")
    print(f"📊 Summary voor tabel: {name}")
    print(f"{'='*40}")
    
    display(df.summary())

In [0]:
# Check timelines
# Print the header for the summary table
print(f"{'DataFrame Name':<20} | {'Oldest Date':<15} | {'Newest Date':<15}")
print("-" * 55)

for name, df in dfs.items():
    if "rapportagedatum" in df.columns:
        # Extract the min and max dates directly to the driver
        stats = df.select(
            F.min("rapportagedatum").cast("string"), 
            F.max("rapportagedatum").cast("string")
        ).collect()[0]
        
        # Print the formatted results
        print(f"{name:<20} | {str(stats[0]):<15} | {str(stats[1]):<15}")
    else:
        # Fallback if the expected column is missing
        print(f"{name:<20} | Column 'rapportagedatum' not found")

**Step 7:** Add a column with adjusted monetary values based on the prices of the tariff list from 2026

In [0]:
# Lookup tabel met alleen prestatiecode en tarief
tarief_lookup = tariffs_2026_df.select("prestatiecode", "tarief_prijspeil_2026")

# Left join voor elke dataset op declaratiecode = prestatiecode
for name, df in [("df_activiteiten", df_activiteiten), ("df_ozp", df_ozp), ("df_verblijf", df_verblijf)]:
    # Drop bestaande kolom(men) om dubbele namen te voorkomen
    df = df.drop("tarief_prijspeil_2026")
    
    joined = df.join(
        tarief_lookup,
        df["declaratiecode"] == tarief_lookup["prestatiecode"],
        "left"
    ).drop("prestatiecode")
    
    # Vul ontbrekende tarieven met 0
    joined = joined.fillna(0, subset=["tarief_prijspeil_2026"])
    
    # Wijs het resultaat terug aan de juiste variabele
    globals()[name] = joined
    
    match_count = joined.filter(F.col("tarief_prijspeil_2026") != 0).count()
    total_count = joined.count()
    print(f"{name}: {match_count}/{total_count} rijen gematcht met tarief")

print("\nKolom 'tarief_prijspeil_2026' toegevoegd aan df_activiteiten, df_ozp en df_verblijf")

In [0]:
dfs = {
    "Activiteiten": df_activiteiten,
    "OZP": df_ozp,
    "Verblijf": df_verblijf,
    "Productiviteit": df_productiviteit,
    "Diensten": df_diensten,
    "tarieven": tariffs_2026_df,
    "fte": df_fte
}

display(dfs['Activiteiten'])

**Step 8:** Export clean tables

In [0]:
import re

# Define the target location in Unity Catalog
catalog_name = "ggzingeest_test"
schema_name = "default"

# Karakters die niet toegestaan zijn in Delta kolomnamen
invalid_chars = re.compile(r'[ ,;{}()\n\t=]')

for name, df in dfs.items():
    # Sanitize kolomnamen: vervang ongeldige karakters door underscores
    for col_name in df.columns:
        clean_name = invalid_chars.sub('_', col_name)
        if clean_name != col_name:
            df = df.withColumnRenamed(col_name, clean_name)
            print(f"  Renamed '{col_name}' -> '{clean_name}'")

    # Construct a clean table name (lowercase, no spaces)
    clean_table_name = f"cleaned_{name.lower().replace(' ', '_')}"
    
    # Create the full 3-level namespace path
    full_table_path = f"{catalog_name}.{schema_name}.{clean_table_name}"
    
    print(f"Registering {name} as a table in Catalog: {full_table_path}...")
    
    # Save the DataFrame as a managed Delta table
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(full_table_path))

print("\n✅ All DataFrames are now successfully overwritten with the new schema!")